## Imports

In [ ]:
from pathlib import Path
import numpy as np
import xarray as xr
import shutil

from setup_lookup_table import create_lookup_table
from flood_adapt.objects.forcing.unit_system import UnitTypesLength
from flood_adapt import FloodAdapt
from flood_adapt.config.config import Settings

## Inputs 
Provide the event set and the range of SLR values to be calculated. 

In [ ]:
# FloodAdapt database paths
DATA_DIR = Path(r"c:\Users\athanasi\Github\Database\Working_Databases\Charleston\4_FloodAdapt\Database")
SFINCS_BIN_PATH=Path(r"c:\Users\athanasi\Github\Database\system\win-64\sfincs_v.2.1_alpha\sfincs.exe")
FIAT_BIN_PATH=Path(r"c:\Users\athanasi\Github\Database\system\win-64\fiat_v0.2.1\fiat.exe")

# Database config
site = "charleston_beta_release"
name_event_set = "probabilistic_set" # "probabilistic_set"

# Scenario input parameters
slr = np.linspace(0, 2, 5)
unit = UnitTypesLength.feet
fp_height = 2  # floodproof height in feet
slr

In [ ]:
new_site_name = site + "_ABM_" + name_event_set
# Copy database
if not (DATA_DIR / new_site_name).exists():
    shutil.copytree(DATA_DIR / site, DATA_DIR / new_site_name)

In [ ]:
if name_event_set == "probabilistic_set":
    full_event_set_path = r"c:\Users\athanasi\Github\Database\Working_Databases\Charleston\3_Event_set\probabilistic_set"
    if not (DATA_DIR / new_site_name / "Input" / "events" / name_event_set).exists():
        shutil.copytree(full_event_set_path, DATA_DIR / new_site_name / "Input" / "events" / name_event_set)  

## Initialize FloodAdapt Database

In [ ]:
# Initialize FloodAdapt
settings = Settings(
    DATABASE_ROOT=DATA_DIR,
    DATABASE_NAME=new_site_name,
    SFINCS_BIN_PATH=SFINCS_BIN_PATH,
    FIAT_BIN_PATH=FIAT_BIN_PATH,
    VALIDATE_BINARIES=True,
)
fa = FloodAdapt(database_path=settings.database_path)

## Create event sequence and look up table

To Do: 
- naming of projections and strategies needs to be more unique to avoid confusion when running a different analysis.
- add method to get impacts at the object ID level

In [ ]:
# Lookup table
ds_impacts = create_lookup_table(
        fa, 
        name_event_set, 
        slr=slr, 
        unit = unit, 
        fp_height = fp_height, 
        )


In [ ]:
ds_impacts

In [ ]:
ds_impacts.to_netcdf(f"lookup_table_{new_site_name}.nc")